In [ ]:
# Cell 1 - Build the VLS section 2.7 general schema
from pathlib import Path
from datetime import datetime, timezone
import json
import subprocess
import sys

def ensure_import(import_name, pip_name=None):
    try:
        __import__(import_name)
        return
    except ModuleNotFoundError:
        if pip_name is None:
            pip_name = import_name
    print(f"Installing missing package for {import_name}: {pip_name}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

ensure_import("pandas", "pandas")

import pandas as pd
from IPython.display import display

if "PROJECT_DIR" not in globals():
    PROJECT_DIR = Path.cwd()
    if PROJECT_DIR.name == "notebooks":
        PROJECT_DIR = PROJECT_DIR.parent

VLS_DIR = PROJECT_DIR / "vls"
VLS_OUT_DIR = PROJECT_DIR / "data_subsets" / "vls_section2_7"
VLS_OUT_DIR.mkdir(parents=True, exist_ok=True)

VLS_SCHEMA_PATH = VLS_OUT_DIR / "vls_section2_7_general_schema.csv"
VLS_SCHEMA_MANIFEST_PATH = VLS_OUT_DIR / "vls_section2_7_general_schema_manifest.json"

# source_code is embedded in lnphub_il_id as LNPhub_{source_code}V########.
# The V######## number is global and keeps increasing across 3CR, 4CR, then ML.
VLS_SOURCES = [
    {
        "source_library": "3CR",
        "source_code": 1,
        "path": VLS_DIR / "VL_3CR.csv",
        "local_id_col": "Lipid_Index",
        "smiles_col": "Canonical_Product_SMILES",
    },
    {
        "source_library": "4CR",
        "source_code": 2,
        "path": VLS_DIR / "VL_4CR.csv",
        "local_id_col": "Lipid_Index",
        "smiles_col": "Canonical_Product_SMILES",
    },
    {
        "source_library": "ML",
        "source_code": 3,
        "path": VLS_DIR / "VL_ML.csv",
        "local_id_col": "LIPID_ID",
        "smiles_col": "Canonical_Product_SMILES",
    },
]

missing = [str(item["path"]) for item in VLS_SOURCES if not item["path"].exists()]
if missing:
    raise FileNotFoundError("Missing VLS input file(s):\n" + "\n".join(missing))

if VLS_SCHEMA_PATH.exists():
    VLS_SCHEMA_PATH.unlink()

next_global_index = 1
write_header = True
source_summaries = []
created_at_utc = datetime.now(timezone.utc).isoformat()

for source in VLS_SOURCES:
    source_path = source["path"]
    source_rows = 0
    missing_local_id = 0
    missing_smiles = 0
    print(f"\nCollecting {source['source_library']} from {source_path}")

    reader = pd.read_csv(
        source_path,
        usecols=[source["local_id_col"], source["smiles_col"]],
        chunksize=500_000,
        dtype=str,
        encoding="utf-8-sig",
        low_memory=False,
    )

    for chunk in reader:
        chunk = chunk.rename(
            columns={
                source["local_id_col"]: "il_id",
                source["smiles_col"]: "il_smiles",
            }
        ).copy()
        chunk["il_id"] = chunk["il_id"].fillna("").astype(str).str.strip()
        chunk["il_smiles"] = chunk["il_smiles"].fillna("").astype(str).str.strip()

        n = len(chunk)
        ids = [
            f"LNPhub_{source['source_code']}V{global_index:08d}"
            for global_index in range(next_global_index, next_global_index + n)
        ]
        next_global_index += n

        out = pd.DataFrame({
            "lnphub_il_id": ids,
            "il_id": chunk["il_id"].to_numpy(),
            "il_smiles": chunk["il_smiles"].to_numpy(),
            "source_library": source["source_library"],
            "source_file": source_path.name,
            "source_code": source["source_code"],
        })
        out.to_csv(VLS_SCHEMA_PATH, mode="a", header=write_header, index=False)
        write_header = False

        source_rows += n
        missing_local_id += int((out["il_id"] == "").sum())
        missing_smiles += int((out["il_smiles"] == "").sum())
        print(f"  wrote {source_rows:,} {source['source_library']} rows", flush=True)

    source_summaries.append({
        "source_library": source["source_library"],
        "source_code": source["source_code"],
        "source_file": source_path.name,
        "rows": source_rows,
        "missing_il_id": missing_local_id,
        "missing_il_smiles": missing_smiles,
    })

schema_info = {
    "created_at_utc": created_at_utc,
    "schema_path": str(VLS_SCHEMA_PATH),
    "columns": ["lnphub_il_id", "il_id", "il_smiles", "source_library", "source_file", "source_code"],
    "id_pattern": "LNPhub_{source_code}V{global_8_digit_index}",
    "source_code_map": {"1": "3CR", "2": "4CR", "3": "ML"},
    "chunkable_for_prediction": True,
    "prediction_chunk_size": 100_000,
    "sources": source_summaries,
    "total_rows": int(sum(item["rows"] for item in source_summaries)),
    "file_size_mb": VLS_SCHEMA_PATH.stat().st_size / (1024 ** 2),
}
VLS_SCHEMA_MANIFEST_PATH.write_text(json.dumps(schema_info, indent=2))

schema_preview = pd.read_csv(VLS_SCHEMA_PATH, nrows=5, dtype=str)
source_summary_df = pd.DataFrame(source_summaries)

print("\nSaved VLS general schema")
print(f"Path: {VLS_SCHEMA_PATH}")
print(f"Manifest: {VLS_SCHEMA_MANIFEST_PATH}")
print(f"Rows: {schema_info['total_rows']:,}")
print(f"Size: {schema_info['file_size_mb']:.2f} MB")
print("\nRows by source:")
display(source_summary_df)
print("\nPreview:")
display(schema_preview)

try:
    from google.colab import files
    files.download(str(VLS_SCHEMA_PATH))
    files.download(str(VLS_SCHEMA_MANIFEST_PATH))
except Exception as exc:
    print("Automatic Colab download did not start in this environment.")
    print("You can download the saved files manually from the paths printed above.")
    print(f"Download helper message: {exc}")


In [ ]:
# Cell 2 - Resume-safe 100K chunk prediction with RDKit MLP, GIN, and ChemBERTa-MTR-Frozen
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

def ensure_import(import_name, pip_names=None):
    try:
        __import__(import_name)
        return
    except ModuleNotFoundError:
        pass
    if pip_names is None:
        pip_names = [import_name]
    elif isinstance(pip_names, str):
        pip_names = [pip_names]
    last_error = None
    for pip_name in pip_names:
        print(f"Installing missing package for {import_name}: {pip_name}")
        result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], text=True, capture_output=True)
        if result.returncode != 0:
            last_error = (result.stdout + result.stderr)[-4000:]
            print(last_error)
            continue
        try:
            __import__(import_name)
            return
        except ModuleNotFoundError as exc:
            last_error = repr(exc)
    raise ModuleNotFoundError(f"Could not import {import_name}. Last install error: {last_error}")

ensure_import("pandas", "pandas")
ensure_import("numpy", "numpy")
ensure_import("joblib", "joblib")
ensure_import("pyarrow", "pyarrow")
ensure_import("rdkit", ["rdkit", "rdkit-pypi"])
ensure_import("torch", "torch")
ensure_import("torch_geometric", "torch-geometric")
ensure_import("transformers", "transformers")

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

if "PROJECT_DIR" not in globals():
    PROJECT_DIR = Path.cwd()
    if PROJECT_DIR.name == "notebooks":
        PROJECT_DIR = PROJECT_DIR.parent

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

VLS_OUT_DIR = PROJECT_DIR / "data_subsets" / "vls_section2_7"
VLS_SCHEMA_PATH = VLS_OUT_DIR / "vls_section2_7_general_schema.csv"
if not VLS_SCHEMA_PATH.exists():
    raise FileNotFoundError(f"Missing {VLS_SCHEMA_PATH}. Run Cell 1 first.")

CHUNK_SIZE = 100_000
MAX_CHUNKS = None                  # Set to a small integer for a smoke test, then back to None.
FORCE_RERUN_COMPLETED_CHUNKS = False
FORCE_REENCODE_COMPLETED_CHUNKS = False
DOWNLOAD_EACH_PREDICTION_CHUNK = False
DOWNLOAD_EACH_ENCODING_PARQUET = False
DOWNLOAD_PROGRESS_FILES = False
START_CHUNK_INDEX = 32              # Set to 0 to scan from the beginning; 32 resumes at chunk 00032.

PREDICTION_ROOT = PROJECT_DIR / "results" / "vls_section2_7_predictions"
CHUNK_INPUT_DIR = PREDICTION_ROOT / "chunk_inputs"
ENCODED_CHUNK_DIR = PREDICTION_ROOT / "encoded_chunks"
PREDICTION_CHUNK_DIR = PREDICTION_ROOT / "prediction_chunks"
INVALID_SMILES_DIR = PREDICTION_ROOT / "invalid_smiles"
PROGRESS_PATH = PREDICTION_ROOT / "vls_section2_7_chunk_progress.csv"
SUMMARY_PATH = PREDICTION_ROOT / "vls_section2_7_prediction_run_summary.json"
for folder in [PREDICTION_ROOT, CHUNK_INPUT_DIR, ENCODED_CHUNK_DIR, PREDICTION_CHUNK_DIR, INVALID_SMILES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CONSENSUS_MODEL_DIR = Path(
    "/content/drive/MyDrive/Colab_Notebooks/NUS_Proejcts/Datasets_Preperation/"
    "Nature_Submission/consensus_models"
)

RDKit_MODEL_PATH = CONSENSUS_MODEL_DIR / "RDKit_Descriptors_MLP_smiles_only.joblib"
GIN_MODEL_PATH = CONSENSUS_MODEL_DIR / "gin_smiles_only.pt"
CHEMBERTA_MODEL_PATH = CONSENSUS_MODEL_DIR / "ChemBERTa_MTR_smiles_only.joblib"

for path in [RDKit_MODEL_PATH, GIN_MODEL_PATH, CHEMBERTA_MODEL_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required model artifact: {path}")

print("Consensus model directory:", CONSENSUS_MODEL_DIR)
print("RDKit MLP:", RDKit_MODEL_PATH)
print("GIN:", GIN_MODEL_PATH)
print("ChemBERTa-MTR frozen classifier:", CHEMBERTA_MODEL_PATH)

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
RDLogger.DisableLog("rdApp.error")
from lnphub_oracles.features import valid_smiles_mask
from lnphub_oracles.train_frozen_classification import predict_proba_fixed
from lnphub_oracles.train_tabular_molecular_classification import build_molecular_features
from lnphub_oracles.train_gnn_classification import graph_dataset, predict as gnn_predict
from lnphub_oracles.torch_models import GINClassifier
import torch

CLASS_LABELS = [0, 1, 2]
CLASS_NAMES = {0: "low", 1: "medium", 2: "high"}
PROB_COLUMNS = [f"prob_{CLASS_NAMES[label]}" for label in CLASS_LABELS]

# The consensus model folder contains the model files only, so recreate the same feature-column
# names used during training instead of reading feature_schema.json files.
RDKit_FEATURE_COLUMNS = [f"rdkit_{name}" for name, _ in Descriptors.descList]
CHEMBERTA_VECTOR_COLUMNS = [f"chemberta_mtr_{i:04d}" for i in range(384)]

rdkit_model = joblib.load(RDKit_MODEL_PATH)
chemberta_model = joblib.load(CHEMBERTA_MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gin_checkpoint = torch.load(GIN_MODEL_PATH, map_location=device, weights_only=False)
gin_num_classes = int(gin_checkpoint.get("num_classes", 3))
gin_tabular_dim = int(gin_checkpoint.get("tabular_dim", 0))
gin_model = GINClassifier(tabular_dim=gin_tabular_dim, num_classes=gin_num_classes).to(device)
gin_model.load_state_dict(gin_checkpoint["state_dict"])
gin_model.eval()
print(f"Device for GIN: {device}")

REPO_DIR = PROJECT_DIR / "encoder_repos"
REPO_DIR.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def run_live(cmd, label, extra_env=None):
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONPATH"] = str(SRC_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    if extra_env:
        env.update(extra_env)
    print("$", " ".join(map(str, cmd)), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"{label} failed with exit code {code}")


def normalize_proba(proba, n_classes=3):
    out = np.asarray(proba, dtype=float)
    if out.ndim != 2:
        raise ValueError(f"Expected a 2D probability array, got shape {out.shape}")
    if out.shape[1] < n_classes:
        padded = np.zeros((out.shape[0], n_classes), dtype=float)
        padded[:, :out.shape[1]] = out
        out = padded
    elif out.shape[1] > n_classes:
        out = out[:, :n_classes]
    row_sums = out.sum(axis=1, keepdims=True)
    out = np.divide(out, row_sums, out=np.full_like(out, 1.0 / n_classes), where=row_sums > 0)
    return out


def assign_proba_columns(frame, prefix, proba):
    proba = normalize_proba(proba, len(CLASS_LABELS))
    for j, label in enumerate(CLASS_LABELS):
        frame[f"{prefix}_prob_{CLASS_NAMES[label]}"] = proba[:, j]


def empty_proba(n_rows):
    return np.full((n_rows, len(CLASS_LABELS)), np.nan, dtype=float)


def load_progress():
    if PROGRESS_PATH.exists():
        return pd.read_csv(PROGRESS_PATH)
    return pd.DataFrame(columns=[
        "chunk_index", "row_start", "row_end", "n_rows", "n_valid_smiles", "n_invalid_smiles",
        "status", "prediction_path", "encoding_path", "started_at_utc", "completed_at_utc", "error",
    ])


def save_progress(row):
    progress = load_progress()
    if not progress.empty:
        progress = progress[progress["chunk_index"].astype(int) != int(row["chunk_index"])]
    progress = pd.concat([progress, pd.DataFrame([row])], ignore_index=True, sort=False)
    progress = progress.sort_values("chunk_index").reset_index(drop=True)
    progress.to_csv(PROGRESS_PATH, index=False)


def download_if_possible(path):
    try:
        from google.colab import files
        files.download(str(path))
    except Exception as exc:
        print(f"Automatic download did not start for {path.name}: {exc}")


def chunk_prediction_path(chunk_index, row_start, n_rows):
    row_end = row_start + n_rows - 1
    return PREDICTION_CHUNK_DIR / f"vls_section2_7_predictions_chunk_{chunk_index:05d}_rows_{row_start:09d}_{row_end:09d}.csv"


def encode_chemberta_chunk(cohort_path, encoded_dir, vector_path):
    if vector_path.exists() and not FORCE_REENCODE_COMPLETED_CHUNKS:
        print("Using existing ChemBERTa-MTR vectors:", vector_path)
        return
    cmd = [
        sys.executable, "-u", "-m", "lnphub_oracles.encode_exact_frozen",
        "--cohort", str(cohort_path),
        "--output-dir", str(encoded_dir),
        "--repo-dir", str(REPO_DIR),
        "--methods", "chemberta_mtr",
    ]
    run_live(cmd, "ChemBERTa-MTR chunk encoding", {"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})


def predict_one_chunk(chunk, chunk_index, row_start):
    chunk = chunk.reset_index(drop=True).copy()
    n_rows = len(chunk)
    row_end = row_start + n_rows - 1
    pred_path = chunk_prediction_path(chunk_index, row_start, n_rows)
    encoded_dir = ENCODED_CHUNK_DIR / f"chunk_{chunk_index:05d}"
    encoded_dir.mkdir(parents=True, exist_ok=True)
    chunk_input_path = CHUNK_INPUT_DIR / f"vls_section2_7_input_chunk_{chunk_index:05d}.csv"
    cohort_path = encoded_dir / f"vls_section2_7_chemberta_cohort_chunk_{chunk_index:05d}.csv"
    vector_path = encoded_dir / "ChemBERTa_MTR_encoder_vectors.parquet"

    if pred_path.exists() and not FORCE_RERUN_COMPLETED_CHUNKS:
        print(f"Skipping completed chunk {chunk_index:05d}: {pred_path}")
        return "skipped", pred_path, vector_path

    started_at = now_utc()
    save_progress({
        "chunk_index": chunk_index,
        "row_start": row_start,
        "row_end": row_end,
        "n_rows": n_rows,
        "status": "running",
        "prediction_path": str(pred_path),
        "encoding_path": str(vector_path),
        "started_at_utc": started_at,
        "completed_at_utc": "",
        "error": "",
    })

    chunk.to_csv(chunk_input_path, index=False)
    valid_mask = valid_smiles_mask(chunk["il_smiles"].fillna("").astype(str))
    valid_positions = np.flatnonzero(valid_mask)
    invalid_positions = np.flatnonzero(~valid_mask)

    predictions = chunk[["lnphub_il_id", "il_id", "il_smiles", "source_library", "source_file", "source_code"]].copy()
    predictions["smiles_valid"] = valid_mask
    predictions["prediction_error"] = np.where(valid_mask, "", "invalid_smiles")

    if len(invalid_positions) > 0:
        invalid_path = INVALID_SMILES_DIR / f"vls_section2_7_invalid_smiles_chunk_{chunk_index:05d}.csv"
        bad_rows = predictions.loc[invalid_positions, [
            "lnphub_il_id", "il_id", "il_smiles", "source_library", "source_file", "source_code", "prediction_error"
        ]].copy()
        bad_rows.insert(0, "chunk_row_position", invalid_positions)
        bad_rows.insert(0, "global_row_position", row_start + invalid_positions)
        bad_rows.to_csv(invalid_path, index=False)
        print(f"Chunk {chunk_index:05d}: found {len(invalid_positions):,} invalid SMILES; saved audit file: {invalid_path}")

    for prefix in ["rdkit_descriptors_mlp", "gin", "chemberta_mtr_frozen"]:
        for label in CLASS_LABELS:
            predictions[f"{prefix}_prob_{CLASS_NAMES[label]}"] = np.nan

    if len(valid_positions) > 0:
        valid_chunk = chunk.loc[valid_mask].reset_index(drop=True).copy()
        valid_chunk["record_id"] = valid_chunk["lnphub_il_id"]

        print(f"Chunk {chunk_index:05d}: RDKit descriptors + MLP on {len(valid_chunk):,} valid SMILES")
        X_rdkit, rdkit_cols = build_molecular_features(valid_chunk["il_smiles"], "rdkit_descriptors")
        if rdkit_cols != RDKit_FEATURE_COLUMNS:
            raise ValueError("RDKit descriptor feature columns do not match the saved model schema.")
        rdkit_proba_valid = predict_proba_fixed(rdkit_model, X_rdkit, CLASS_LABELS)
        rdkit_proba = empty_proba(n_rows)
        rdkit_proba[valid_positions, :] = normalize_proba(rdkit_proba_valid)
        for j, label in enumerate(CLASS_LABELS):
            predictions.loc[:, f"rdkit_descriptors_mlp_prob_{CLASS_NAMES[label]}"] = rdkit_proba[:, j]

        print(f"Chunk {chunk_index:05d}: GIN on {len(valid_chunk):,} valid SMILES")
        gin_input = valid_chunk.copy()
        gin_input["_orig_pos"] = valid_positions
        gin_input["delivery_class_3"] = 0
        graphs, gin_frame = graph_dataset(gin_input, "delivery_class_3", None)
        gin_proba = empty_proba(n_rows)
        if graphs:
            _, _, gin_proba_graphs, row_idx = gnn_predict(gin_model, graphs, type("Args", (), {"batch_size": 512})(), device)
            orig_pos = gin_frame.iloc[row_idx]["_orig_pos"].to_numpy(dtype=int)
            gin_proba[orig_pos, :] = normalize_proba(gin_proba_graphs)
        missing_graphs = sorted(set(valid_positions.tolist()) - set(np.flatnonzero(np.isfinite(gin_proba).all(axis=1)).tolist()))
        if missing_graphs:
            predictions.loc[missing_graphs, "prediction_error"] = predictions.loc[missing_graphs, "prediction_error"].mask(
                predictions.loc[missing_graphs, "prediction_error"].eq(""), "gin_graph_failed"
            )
        for j, label in enumerate(CLASS_LABELS):
            predictions.loc[:, f"gin_prob_{CLASS_NAMES[label]}"] = gin_proba[:, j]

        print(f"Chunk {chunk_index:05d}: ChemBERTa-MTR frozen encoding + classifier")
        cohort = valid_chunk[["record_id", "il_id", "il_smiles"]].copy()
        cohort.to_csv(cohort_path, index=False)
        encode_chemberta_chunk(cohort_path, encoded_dir, vector_path)
        vectors = pd.read_parquet(vector_path)
        missing_vector_cols = [col for col in CHEMBERTA_VECTOR_COLUMNS if col not in vectors.columns]
        if missing_vector_cols:
            raise ValueError(f"ChemBERTa vector file is missing expected columns, first few: {missing_vector_cols[:5]}")
        aligned = chunk[["lnphub_il_id"]].rename(columns={"lnphub_il_id": "record_id"}).merge(
            vectors[["record_id"] + CHEMBERTA_VECTOR_COLUMNS], on="record_id", how="left", validate="one_to_one"
        )
        vector_ok = aligned[CHEMBERTA_VECTOR_COLUMNS].notna().all(axis=1).to_numpy()
        chemberta_proba = empty_proba(n_rows)
        if vector_ok.any():
            X_chemberta = aligned.loc[vector_ok, CHEMBERTA_VECTOR_COLUMNS].to_numpy(dtype=np.float32)
            chemberta_proba[vector_ok, :] = normalize_proba(predict_proba_fixed(chemberta_model, X_chemberta, CLASS_LABELS))
        for j, label in enumerate(CLASS_LABELS):
            predictions.loc[:, f"chemberta_mtr_frozen_prob_{CLASS_NAMES[label]}"] = chemberta_proba[:, j]
    else:
        print(f"Chunk {chunk_index:05d}: no valid SMILES; writing invalid-only prediction file")

    temp_path = pred_path.with_suffix(".tmp.csv")
    predictions.to_csv(temp_path, index=False)
    temp_path.replace(pred_path)

    row = {
        "chunk_index": chunk_index,
        "row_start": row_start,
        "row_end": row_end,
        "n_rows": n_rows,
        "n_valid_smiles": int(valid_mask.sum()),
        "n_invalid_smiles": int((~valid_mask).sum()),
        "status": "complete",
        "prediction_path": str(pred_path),
        "encoding_path": str(vector_path) if vector_path.exists() else "",
        "started_at_utc": started_at,
        "completed_at_utc": now_utc(),
        "error": "",
    }
    save_progress(row)
    print(f"Saved chunk {chunk_index:05d}: {pred_path}")
    if DOWNLOAD_EACH_PREDICTION_CHUNK:
        download_if_possible(pred_path)
    if DOWNLOAD_EACH_ENCODING_PARQUET and vector_path.exists():
        download_if_possible(vector_path)
    return "complete", pred_path, vector_path


run_started_at = now_utc()
print(f"Starting VLS predictions at {run_started_at}")
print(f"Input: {VLS_SCHEMA_PATH}")
print(f"Chunk size: {CHUNK_SIZE:,}")
print(f"Progress file: {PROGRESS_PATH}")

processed_this_run = []
reader = pd.read_csv(VLS_SCHEMA_PATH, chunksize=CHUNK_SIZE, dtype=str, low_memory=False)
for chunk_index, chunk in enumerate(reader):
    if chunk_index < START_CHUNK_INDEX:
        continue
    if MAX_CHUNKS is not None and chunk_index >= MAX_CHUNKS:
        print(f"Stopping after MAX_CHUNKS={MAX_CHUNKS}")
        break
    row_start = chunk_index * CHUNK_SIZE
    try:
        status, pred_path, vector_path = predict_one_chunk(chunk, chunk_index, row_start)
        processed_this_run.append({"chunk_index": chunk_index, "status": status, "prediction_path": str(pred_path)})
    except Exception as exc:
        row_end = row_start + len(chunk) - 1
        save_progress({
            "chunk_index": chunk_index,
            "row_start": row_start,
            "row_end": row_end,
            "n_rows": len(chunk),
            "n_valid_smiles": "",
            "n_invalid_smiles": "",
            "status": "failed",
            "prediction_path": str(chunk_prediction_path(chunk_index, row_start, len(chunk))),
            "encoding_path": str(ENCODED_CHUNK_DIR / f"chunk_{chunk_index:05d}" / "ChemBERTa_MTR_encoder_vectors.parquet"),
            "started_at_utc": "",
            "completed_at_utc": now_utc(),
            "error": repr(exc),
        })
        print(f"Chunk {chunk_index:05d} failed. Progress was saved so the next run can resume.")
        raise

progress = load_progress()
summary = {
    "run_started_at_utc": run_started_at,
    "run_finished_at_utc": now_utc(),
    "input_schema_path": str(VLS_SCHEMA_PATH),
    "chunk_size": CHUNK_SIZE,
    "prediction_root": str(PREDICTION_ROOT),
    "progress_path": str(PROGRESS_PATH),
    "n_progress_rows": int(len(progress)),
    "n_complete_chunks": int((progress["status"] == "complete").sum()) if not progress.empty else 0,
    "n_failed_chunks": int((progress["status"] == "failed").sum()) if not progress.empty else 0,
    "processed_this_run": processed_this_run,
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))

print("\nPrediction run summary")
print(json.dumps(summary, indent=2))
print("\nProgress preview:")
display(progress.tail(10))
if DOWNLOAD_PROGRESS_FILES:
    try:
        from google.colab import files
        files.download(str(PROGRESS_PATH))
        files.download(str(SUMMARY_PATH))
    except Exception as exc:
        print("Automatic download did not start for progress/summary files.")
        print(f"Download helper message: {exc}")
else:
    print("Automatic progress/summary downloads are disabled. Files are saved at the paths printed above.")
